# DDL

**Data Definition Language** (DDL) is a subset of SQL commands that can be used to define the entities in a database. This page considers important DDL structures.

## Keys

The tables allows you to define relationships with other tables. This is achieved through defining:

- **Primary key**: a column or set of columns whose values **uniquely identify each row** in a table. A primary key must be unique and normally cannot be NULL. It is the main identifier of a row and is used to distinguish one record from all others.
- **Foreign key** : a column or set of columns in one table whose values **reference the primary key**  of another table. It defines a relationship between tables and enforces referential integrity: a foreign-key value must either match an existing referenced row (or be NULL if allowed).

---

The following cell creates the tables `users` and `orders` and defines `users.id` as a primary key and relates the `orders.user_id` to the `users.id`.

In [39]:
--postgreSQL
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS users;

CREATE TABLE users(
    id INT PRIMARY KEY
);

CREATE TABLE orders (
    id INT PRIMARY KEY,
    user_id INT REFERENCES users(id)
);
COMMIT;

NOTICE: table "orders" does not exist, skipping
NOTICE: table "users" does not exist, skipping
DROP TABLE
DROP TABLE
CREATE TABLE
CREATE TABLE
COMMIT


The following cell attepts to insert a record into the `orders` table even though no order has been created.

In [ ]:
--postgreSQL
INSERT INTO orders VALUES (1, 1);

Traceback (most recent call last):
  File "/home/fedor/Documents/code/knowledge/src/sql_kernel/kernel.py", line 116, in do_execute
    self._execute_sql(code=code)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^
  File "/home/fedor/Documents/code/knowledge/src/sql_kernel/kernel.py", line 84, in _execute_sql
    messages, tables = runner.execute(code)
                       ~~~~~~~~~~~~~~^^^^^^
  File "/home/fedor/Documents/code/knowledge/src/sql_kernel/runners/runners.py", line 73, in execute
    raise e
  File "/home/fedor/Documents/code/knowledge/src/sql_kernel/runners/runners.py", line 70, in execute
    cursor.execute(code.encode())
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^
  File "/home/fedor/.virtualenvrionements/knowledge/lib/python3.13/site-packages/psycopg/cursor.py", line 97, in execute
    raise ex.with_traceback(None)
psycopg.errors.ForeignKeyViolation: insert or update on table "orders" violates foreign key constraint "orders_user_id_fkey"
DETAIL:  Key (user_id)=(1) is not present in table "user

However, when the corresponding ID  has already been created, everything goes smoothly:

In [41]:
--postgreSQL
INSERT INTO users VALUES (1);
INSERT INTO orders VALUES (1, 1);

INSERT 0 1
INSERT 0 1


## Default values

When defining a table in SQL, the default value can be specified. Typically, the keyword `DEFAULT` indicates that the default value will follow. If values for columns with a default are not specified when inserting, the deafults will be used.

---

The following cell creates table with default specified to both columns.

In [3]:
--postgreSQL
DROP TABLE IF EXISTS table_with_default; 
CREATE TABLE table_with_default (
    col1 INT DEFAULT 10,
    col2 VARCHAR DEFAULT 'default'
);

NOTICE: table "table_with_default" does not exist, skipping
DROP TABLE
CREATE TABLE


The following cell makes two inserts: the first specifies only `col1`, the second only `col2`.

In [4]:
--postgreSQL
INSERT INTO table_with_default (col1)
VALUES (1);

INSERT INTO table_with_default (col2)
VALUES ('hello');

SELECT * FROM table_with_default;

INSERT 0 1
INSERT 0 1
SELECT 2


col1,col2
1,default
10,hello


Consequently, the rows for those columns that are not specified are filled with default values.

## Alter table

The `ALTER TABLE` is a command to change the definition of the existing table. The popular sematics for most SQL dialects is:

```sql
ALTER TABLE table_name
    [[ADD|DROP] [COLUMN|CONSTRAINT] | ALTER COLUMN]
```

---

The following cell creates the table with some columns:

In [6]:
--postgreSQL
DROP TABLE IF EXISTS some_table;
CREATE TABLE some_table AS
SELECT * FROM (
    VALUES 
        (10, 'hello'),
        (20, 'new value')
) AS t(col1, col2);

NOTICE: table "some_table" does not exist, skipping
DROP TABLE
SELECT 2


Some properties of the columns of the table:

In [7]:
--postgreSQL
SELECT
    column_name,
    column_default,
    is_nullable,
    data_type
FROM information_schema.columns
WHERE table_name = 'some_table';

SELECT 2


column_name,column_default,is_nullable,data_type
col1,,YES,integer
col2,,YES,text


The following cell uses `ALTER TABLE` command to:

- Add new column with name `new_col`.
- Make the column `col1` not nullable.

In [8]:
--postgreSQL
ALTER TABLE some_table ADD COLUMN new_col INT;
ALTER TABLE some_table ALTER COLUMN col1 SET NOT NULL;

ALTER TABLE
ALTER TABLE


The corresponding changes appear in the `information_schema.columns`:

In [9]:
--postgreSQL
SELECT
    column_name,
    column_default,
    is_nullable,
    data_type
FROM information_schema.columns
WHERE table_name = 'some_table';

SELECT 3


column_name,column_default,is_nullable,data_type
col1,,NO,integer
new_col,,YES,integer
col2,,YES,text
